<a href="https://colab.research.google.com/github/GEO-HACK/watchtower-ml/blob/main/xGboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import joblib
import gc
import matplotlib.pyplot as plt
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.utils.class_weight import compute_sample_weight

print(f"✅ XGBoost version : {xgb.__version__}")
print("✅ All imports done")

Mounted at /content/drive
✅ XGBoost version : 3.2.0
✅ All imports done


In [ ]:
BASE = '/content/drive/MyDrive/CICIDS/'

X_train = np.load(BASE + 'X_train.npy')
X_test  = np.load(BASE + 'X_test.npy')
y_train = np.load(BASE + 'y_train.npy')
y_test  = np.load(BASE + 'y_test.npy')

le       = joblib.load(BASE + 'label_encoder.pkl')
pipeline = joblib.load(BASE + 'preprocessing_pipeline.pkl')


print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

print(f"\nPipeline steps:")
for name, step in pipeline.steps:
    print(f"  → {name} : {step.__class__.__name__}")

print(f"\nClasses ({len(le.classes_)}):")
for idx, cls in enumerate(le.classes_):
    print(f"  {idx} → {cls}")

X_train : (1897363, 70)
X_test  : (474341, 70)
y_train : (1897363,)
y_test  : (474341,)

Pipeline steps:
  → imputer : SimpleImputer
  → scaler : RobustScaler

Classes (10):
  0 → BENIGN
  1 → Bot
  2 → DDoS
  3 → DoS GoldenEye
  4 → DoS Hulk
  5 → DoS Slowhttptest
  6 → DoS slowloris
  7 → FTP-Patator
  8 → PortScan
  9 → SSH-Patator


In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print("✅ Sample weights computed")
print(f"Weight range : {sample_weights.min():.4f} → {sample_weights.max():.4f}")

✅ Sample weights computed
Weight range : 0.1306 → 120.6207


In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=y_train
)

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='multi:softmax',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost...")
xgb_model.fit(
    X_train, y_train,
    sample_weight=sample_weights
)
print("✅ Training complete")

Training XGBoost...
✅ Training complete


In [ ]:
from sklearn.metrics import classification_report, f1_score

y_pred   = xgb_model.predict(X_test)
macro_f1 = f1_score(y_test, y_pred, average='macro')

print("=" * 60)
print("CLASSIFICATION REPORT — XGBoost")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=4))
print(f"Macro F1 : {macro_f1:.4f}")

CLASSIFICATION REPORT — XGBoost
                  precision    recall  f1-score   support

          BENIGN     1.0000    0.9996    0.9998    363257
             Bot     0.8846    0.9949    0.9365       393
            DDoS     0.9996    0.9999    0.9997     25605
   DoS GoldenEye     0.9956    0.9995    0.9976      2059
        DoS Hulk     0.9988    0.9998    0.9993     46215
DoS Slowhttptest     0.9937    1.0000    0.9968      1100
   DoS slowloris     0.9974    0.9974    0.9974      1159
     FTP-Patator     0.9994    1.0000    0.9997      1588
        PortScan     0.9999    0.9999    0.9999     31786
     SSH-Patator     1.0000    1.0000    1.0000      1179

        accuracy                         0.9997    474341
       macro avg     0.9869    0.9991    0.9927    474341
    weighted avg     0.9997    0.9997    0.9997    474341

Macro F1 : 0.9927


In [ ]:
joblib.dump(xgb_model, BASE + 'xgboost_model.pkl')
print("✅ xgboost_model.pkl saved to Drive")

✅ xgboost_model.pkl saved to Drive
